In [ ]:
#| default_exp solve_auth

# SolveIt Auth

> Google Sign-In for apps hosted on SolveIt

This module wraps the SolveIt auth service so your FastHTML app can add Google Sign-In with just two function calls: `setup_solve_signin` to add the sign-in route, and `sub_from_signin` to validate the reply and get the user's Google sub ID.

## Setup -

In [ ]:
#| export
import os, json, httpx, jwt

In [ ]:
#| export
class SolveSigninError(Exception): pass

_AUTH_URL = 'https://auth.solve.it.com/request_signin'

def _key(): return os.environ['AAI_USER_KEY']
def _app_url(port): return f'https://{json.loads(os.environ["PUBLIC_DOMAINS"])[str(port)]}'

`SolveSigninError` is raised when authentication fails — either from an invalid JWT or an error returned by the auth service. Catch this in your `signin_completed` route to handle auth failures gracefully.

In [ ]:
#| export
solve_signin_rt = '/solve_signin'
def setup_solve_signin(app, port=8000, callback_rt='/signin_completed', email_re = None, hd_re = None):
    "Add /solve_signin route to `app` that redirects to Google via SolveIt auth"
    from starlette.responses import RedirectResponse
    callback = _app_url(port) + callback_rt
    @app.route(solve_signin_rt)
    def solve_signin():
        r = httpx.post(_AUTH_URL, headers={'Authorization': _key()}, json={'callback_url': callback, 'email_re': email_re, 'hd_re':hd_re})
        return RedirectResponse(r.json()['signin_url'])

`setup_solve_signin` adds a `/solve_signin` route to your FastHTML app. When a user visits this route, they're redirected to Google Sign-In via the SolveIt auth service. After authentication, Google redirects back to your `callback_rt` (defaults to `/signin_completed`) with a `signin_reply` query parameter.

The `solve_signin_rt` variable is exported so you can reference it in links: `A('Sign In', href=solve_signin_rt)`.

In [ ]:
#| export
def sub_from_signin(session, signin_reply):
    "Decode signin JWT, validate, and return Google sub ID. Raises SolveSigninError on failure."
    try:
        payload = jwt.decode(signin_reply, _key(), algorithms=['HS256'])
        if 'err' in payload: raise SolveSigninError(payload['err'])
        return str(payload['sub'])
    except jwt.PyJWTError as e: raise SolveSigninError(f'JWT validation failed: {e}') from e

`sub_from_signin` is the function your app calls in the signin completion route. It decodes the JWT reply from SolveIt's auth service, checks for errors, and returns the user's Google sub ID as a string. Raises `SolveSigninError` on any failure.

## Manual Test

In this section we create an app running on port 8000 that uses SolveIt Auth.

In [ ]:
from dialoghelper.solve_auth import *
from fasthtml.common import *
from fasthtml.jupyter import *

In [ ]:
app_url = "http://localhost:8000"

`.solve_auth._app_url` uses https. We patch it so that we can test locally with http.

In [ ]:
import dialoghelper

In [ ]:
def _app_url(port): return app_url
dialoghelper.solve_auth._app_url = _app_url

In [ ]:
app, rt = fast_app()
srv = JupyUvi(app, port=8000)
setup_solve_signin(app)

In [ ]:
@rt
def index(): return Titled('Auth Test', A('Sign In with Google', href=solve_signin_rt, cls='button'))

@rt
def signin_completed(session, signin_reply: str=''):
    session['auth'] = sub_from_signin(session, signin_reply)
    return Titled('Signed In!', Pre('Sub ID: ' + session['auth']))

In [ ]:
import warnings

In [ ]:
warnings.filterwarnings("ignore", message=".*HMAC key.*")

We get a warning if our `AAI_USER_KEY` is too short. We suppress that warning here so that our notebook diffs are clean when running our tests locally.

In [ ]:
A("Try it out", href=app_url, target='_blank')

<a href="http://localhost:8000" target="_blank">Try it out</a>

Click the link above, run through the Google Sign and confirm that your Google sub id is displayed on redirect.

In [ ]:
srv.stop()

## Export -

In [ ]:
#| hide
from nbdev import nbdev_export
nbdev_export()